# Social Media Sentiment Analysis

This notebook runs the whole project in one place.

Main idea:
- we clean the original comments
- we train machine learning models on labeled comments
- we keep the best model
- we test it on new comments
- we compare it with the VADER baseline


## Notebook Flow

What each part means:
- **Raw data**: the original comments dataset
- **Preprocessed data**: cleaned comments ready for ML
- **Training**: the model learns patterns from labeled examples
- **Prediction**: we give new comments and ask for sentiment labels
- **Sample demo**: prepared example posts used to show the system working end-to-end


In [1]:
from pathlib import Path
import json
import sys

# Make sure the notebook can find the project's Python files.
project_root = Path.cwd().resolve()
if not (project_root / 'src').exists():
    project_root = project_root.parent

src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print('Project root:', project_root)

Project root: C:\Me\fac-master\social-media-sentiment-analysis


In [2]:
# Import the functions we will use throughout the notebook.
from common import PROCESSED_DATA_PATH, MODELS_DIR, RAW_DATA_PATH, read_sample_posts
from preprocess import preprocess_dataset
from train_ml import train_models
from predict_pretrained import predict_comments_with_vader
from predict_ml import predict_comments

## 1. Preprocess The Dataset

Here we take the raw comments and clean them.

Examples of preprocessing:
- lowercasing text
- removing links and punctuation
- removing extra spaces

This gives us **preprocessed data**, which is simply a cleaner version of the original comments.

In [3]:
# Run preprocessing and create the cleaned dataset used by training.
count = preprocess_dataset()
print(f'Preprocessed {count} comments')
print('Raw dataset:', RAW_DATA_PATH)
print('Processed dataset:', PROCESSED_DATA_PATH)

Preprocessed 45 comments
Raw dataset: C:\Me\fac-master\social-media-sentiment-analysis\data\raw\comments.csv
Processed dataset: C:\Me\fac-master\social-media-sentiment-analysis\data\processed\comments_clean.csv


## 2. Train The Machine Learning Models

In this step, the algorithms do not just run rules that we wrote by hand.

Instead, they **learn from labeled examples** in the dataset.

We train several models:
- Logistic Regression
- Linear SVM
- Multinomial Naive Bayes

Then we compare them and keep the best one.

So yes: the **trained ML model** is one of those models after it has learned from the dataset.

In [4]:
# Train the candidate models and keep the best one.
training_result = train_models()

print('Best model:', training_result['summary']['best_model'])
print('Accuracy:', round(training_result['metrics']['accuracy'], 3))
print('Macro F1:', round(training_result['metrics']['macro_f1'], 3))

Best model: linear_svm
Accuracy: 0.429
Macro F1: 0.444


In [5]:
# Show the training summary in a readable way.
print(json.dumps(training_result['summary'], indent=2))

{
  "best_model": "linear_svm",
  "leaderboard": [
    {
      "model": "linear_svm",
      "validation_macro_f1": 0.4000000000000001
    },
    {
      "model": "logistic_regression",
      "validation_macro_f1": 0.13333333333333333
    },
    {
      "model": "multinomial_nb",
      "validation_macro_f1": 0.13333333333333333
    }
  ],
  "data_path": "C:\\Me\\fac-master\\social-media-sentiment-analysis\\data\\processed\\comments_clean.csv",
  "saved_model_path": "C:\\Me\\fac-master\\social-media-sentiment-analysis\\models\\best_model.pkl"
}


## 3. Test On New Comments

Now we give the system comments that are not the training labels we used to teach it.

This is the important idea:
- **Training** = the model learns from labeled examples
- **Testing / prediction** = we give new comments and see what it predicts

So yes, this section is where we test the trained model on comments we type here.

In [6]:
# Change these comments whenever you want to test your own examples.
custom_comments = [
    'I love this update',
    'Pretty decent overall',
    'Not good at all'
]

custom_comments

['I love this update', 'Pretty decent overall', 'Not good at all']

### 3A. VADER Baseline

This is the existing sentiment tool baseline.

In [7]:
# Run VADER on the custom comments.
vader_result = predict_comments_with_vader(custom_comments)

for comment, label, score in zip(vader_result['comments'], vader_result['labels'], vader_result['scores']):
    print(f'[{label}] compound={score["compound"]:.3f} :: {comment}')

print(vader_result['reaction'])

[positive] compound=0.637 :: I love this update
[positive] compound=0.494 :: Pretty decent overall
[negative] compound=-0.341 :: Not good at all
{'reaction': 'Mixed Reaction', 'counts': {'positive': 2, 'negative': 1, 'neutral': 0, 'total': 3}}


### 3B. Trained ML Model

This uses the best model learned from the dataset.

In [8]:
# Run the trained ML model on the same custom comments.
ml_result = predict_comments(custom_comments)

for comment, label in zip(ml_result['comments'], ml_result['labels']):
    print(f'[{label}] {comment}')

print(ml_result['reaction'])

[positive] I love this update
[negative] Pretty decent overall
[negative] Not good at all
{'reaction': 'Mixed Reaction', 'counts': {'positive': 1, 'negative': 2, 'neutral': 0, 'total': 3}}


## 4. Sample Demo

The sample demo is just a few prepared posts with comment lists.

We use them to demonstrate the full system clearly during testing or presentation.

In [9]:
# Load the prepared example posts.
sample_posts = read_sample_posts()
sample_posts

[{'post_id': 'post_1',
  'title': 'Highly positive launch reactions',
  'comments': ['Love this release, great job!',
   'This is amazing and super helpful.',
   'Very clean work, I enjoyed it.',
   'Nice improvements overall.',
   'Honestly this is excellent.']},
 {'post_id': 'post_2',
  'title': 'Mostly negative reactions',
  'comments': ['This update is frustrating.',
   'I do not like the new direction.',
   'Bad experience for me.',
   'Confusing and messy.',
   'Please undo this change.']},
 {'post_id': 'post_3',
  'title': 'Mixed audience response',
  'comments': ['Great visuals but the explanation is weak.',
   'I like parts of it.',
   'It is okay, nothing special.',
   'Not very useful for me.',
   'Some good ideas, some bad ones.']}]

In [10]:
# Run the trained ML model on each prepared sample post.
for post in sample_posts:
    print('\n' + '=' * 60)
    print(post['post_id'], '-', post['title'])
    print('-' * 60)
    result = predict_comments(post['comments'])
    for comment, label in zip(post['comments'], result['labels']):
        print(f'[{label}] {comment}')
    print(result['reaction'])


post_1 - Highly positive launch reactions
------------------------------------------------------------
[positive] Love this release, great job!
[positive] This is amazing and super helpful.
[positive] Very clean work, I enjoyed it.
[negative] Nice improvements overall.
[positive] Honestly this is excellent.
{'reaction': 'Liked', 'counts': {'positive': 4, 'negative': 1, 'neutral': 0, 'total': 5}}

post_2 - Mostly negative reactions
------------------------------------------------------------
[positive] This update is frustrating.
[negative] I do not like the new direction.
[neutral] Bad experience for me.
[negative] Confusing and messy.
[negative] Please undo this change.
{'reaction': 'Not Liked', 'counts': {'positive': 1, 'negative': 3, 'neutral': 1, 'total': 5}}

post_3 - Mixed audience response
------------------------------------------------------------
[positive] Great visuals but the explanation is weak.
[positive] I like parts of it.
[neutral] It is okay, nothing special.
[neutr

## 5. Files Created By The Notebook

In [11]:
# Show the important output files created during preprocessing and training.
print('Processed dataset:', PROCESSED_DATA_PATH)
print('Models folder:', MODELS_DIR)

for path in sorted(MODELS_DIR.glob('*')):
    print('-', path.name)

Processed dataset: C:\Me\fac-master\social-media-sentiment-analysis\data\processed\comments_clean.csv
Models folder: C:\Me\fac-master\social-media-sentiment-analysis\models
- .gitkeep
- best_model.pkl
- confusion_matrix.png
- test_metrics.json
- training_summary.json
